# Dashboard Formula Inference

## Objetivo

Inferir automaticamente quais indicadores provavelmente compõem
os cálculos exibidos em um dashboard utilizando engenharia reversa.

---

## Contexto

Nem sempre as fórmulas utilizadas em dashboards estão documentadas.

Este projeto automatiza a busca por possíveis combinações entre
indicadores originais capazes de reproduzir os valores apresentados.

---

## Metodologia

1. Leitura dos dados
2. Construção do dicionário de indicadores
3. Geração das combinações
4. Cálculo das razões
5. Comparação com o dashboard
6. Ranqueamento das melhores aproximações

---

## Tecnologias

Python

Pandas

Itertools

Regex

OpenPyXL

> **Note:** Os dados originais foram removidos por questões de confidencialidade.Este notebook preserva toda a metodologia empregada.


In [ ]:
# =============================================================================from pathlib import Path
import itertools
import logging
import re

import pandas as pd

In [ ]:
# =============================================================================
# CONFIGURAÇÕES
# =============================================================================

ARQUIVO_DASHBOARD = Path("dados/dashboard.xlsx")
ARQUIVO_INDICADORES = Path("dados/indicadores.xlsx")

PASTA_SAIDA = Path("output")
ARQUIVO_SAIDA = "dashboard_formula_analysis.xlsx"


logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(message)s"
)

In [ ]:
# =============================================================================
# FUNÇÕES AUXILIARES
# =============================================================================

def extrair_multiplicador(texto) -> int:
    """
    Extrai o multiplicador informado na planilha.

    Exemplos
    --------
    x100 -> 100
    x1000 -> 1000
    NaN -> 1
    """

    if not isinstance(texto, str):
        return 1

    match = re.search(r"x(\d+)", texto)

    if match:
        return int(match.group(1))

    return 1


def criar_dicionario_indicadores(df: pd.DataFrame) -> dict:
    """
    Converte a tabela de indicadores em um dicionário.

    Returns
    -------
    dict

    Exemplo

    {
        "Óbitos": 120,
        "Internações": 340
    }
    """

    return dict(zip(df["Indicador"], df["Resultado"]))


def gerar_combinacoes(indicadores: dict):
    """
    Gera todas as combinações possíveis entre indicadores.

    Utilizamos permutations em vez de combinations porque:

        A / B != B / A
    """

    return itertools.permutations(indicadores.items(), 2)




In [ ]:
# =============================================================================
# PROCESSAMENTO
# =============================================================================

def calcular_combinacoes(
    dashboard: pd.DataFrame,
    indicadores: dict
) -> pd.DataFrame:
    """
    Testa todas as combinações possíveis de divisão entre indicadores.
    """

    resultados = []

    combinacoes = list(gerar_combinacoes(indicadores))

    logging.info(
        "Total de combinações possíveis: %s",
        len(combinacoes)
    )

    for linha in dashboard.itertuples(index=False):

        metrica = linha.Métrica
        valor_dashboard = linha._1 if False else getattr(linha, "Meu 2024")
        multiplicador = getattr(linha, "Multiplica")

        if pd.isna(valor_dashboard):
            continue

        multiplicador = extrair_multiplicador(multiplicador)

        for (nome_num, valor_num), (nome_den, valor_den) in combinacoes:

            if valor_den == 0:
                continue

            resultado = (valor_num / valor_den) * multiplicador

            diferenca = abs(resultado - valor_dashboard)

            resultados.append({

                "Métrica": metrica,

                "Numerador": nome_num,

                "Denominador": nome_den,

                "Resultado Calculado": resultado,

                "Valor Dashboard": valor_dashboard,

                "Multiplicador": multiplicador,

                "Diferença": diferenca

            })

    return pd.DataFrame(resultados)




In [ ]:
# =============================================================================
# CLASSIFICAÇÃO
# =============================================================================

def classificar_resultados(df: pd.DataFrame) -> pd.DataFrame:
    """
    Classifica os resultados por proximidade.
    """

    df["Ranking"] = (

        df
        .groupby("Métrica")["Diferença"]
        .rank(method="dense")

    )

    df["Melhor Aproximação"] = (

        df["Ranking"] == 1

    )

    df["Pior Aproximação"] = (

        df.groupby("Métrica")["Diferença"]
        .transform(lambda x: x == x.max())

    )

    return df.sort_values(

        ["Métrica", "Ranking", "Diferença"]

    )




In [ ]:
# =============================================================================
# EXPORTAÇÃO
# =============================================================================

def salvar_resultados(
    resultados: pd.DataFrame,
    pasta_saida: Path,
    nome_arquivo: str
):
    """
    Exporta os resultados para Excel.

    São criadas duas abas:

    1. Todas as combinações.
    2. Top 10 aproximações por métrica.
    """

    pasta_saida.mkdir(exist_ok=True)

    caminho = pasta_saida / nome_arquivo

    top10 = (

        resultados

        .sort_values(["Métrica", "Diferença"])

        .groupby("Métrica")

        .head(10)

    )

    with pd.ExcelWriter(caminho) as writer:

        resultados.to_excel(

            writer,

            sheet_name="Todas as combinações",

            index=False

        )

        top10.to_excel(

            writer,

            sheet_name="Top 10",

            index=False

        )

    logging.info("Arquivo salvo em:")

    logging.info(caminho.resolve())




In [ ]:
# =============================================================================
# EXECUÇÃO
# =============================================================================

def main():
    """
    Fluxo principal do projeto.
    """

    # -------------------------------------------------------------------------
    # LEITURA DOS DADOS
    # -------------------------------------------------------------------------
    #
    # Os arquivos utilizados originalmente foram removidos por questões de
    # confidencialidade.
    #
    # Espera-se:
    #
    # dashboard.xlsx
    #
    # Métrica
    # Meu 2024
    # Multiplica
    #
    #
    # indicadores.xlsx
    #
    # Indicador
    # Resultado
    #
    # -------------------------------------------------------------------------

    logging.info("Lendo arquivos...")

    dashboard = pd.read_excel(ARQUIVO_DASHBOARD)

    indicadores_df = pd.read_excel(ARQUIVO_INDICADORES)

    indicadores = criar_dicionario_indicadores(indicadores_df)

    logging.info(
        "%s indicadores carregados.",
        len(indicadores)
    )

    resultados = calcular_combinacoes(

        dashboard,

        indicadores

    )

    resultados = classificar_resultados(resultados)

    salvar_resultados(

        resultados,

        PASTA_SAIDA,

        ARQUIVO_SAIDA

    )

    logging.info("Processamento concluído com sucesso!")


if __name__ == "__main__":

    main()